# Data loader for LLMs

In this notebook i am immplementing my dataloader for llms , i am using `byte pair-encoding` for that 

In [ ]:
# importing tiktoken library to implement BPE

import tiktoken
print("version:", tiktoken.__version__)     

version: 0.14.0


Once installed, we can instantiate the BPE tokenizer from tiktoken as follows:

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [5]:
string = tokenizer.decode(integers)
print(string)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [6]:
print("vocab size:", tokenizer.n_vocab)

vocab size: 50257


lets load our dataset and apply and check how many token does it have


## Implementing BPE algorithm on our own dataset

In [9]:
from datasets import load_from_disk

dataset = load_from_disk(
   r"C:\LLM from Scratch\Data"
)

In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

In [ ]:
# using fraction of part of  our dataset
dataset_text = " ".join(dataset["train"]['text'][:20000]) 

In [12]:
encode_dataset = tokenizer.encode(dataset_text, allowed_special={"<|endoftext|>"})

In [13]:
print("total tokens in the dataset:", len(encode_dataset))

total tokens in the dataset: 4443976


In [14]:
# total vocab size of the tokenizer
print("vocab size:", tokenizer.n_vocab)

vocab size: 50257


### How Embedding Models Learn via Next-Word Prediction

Embedding models use **Self-Supervised Learning** to train directly on raw text without human labels:

1. **Input-Target Pairs:** A sliding window splits text into an **Input** (a sequence of words) and a **Target** (the immediate next word).
2. **Prediction:** The model uses its current word embeddings to predict the Target.
3. **Error Calculation:** A loss function measures how far the prediction was from the actual Target.
4. **Weight Update:** Through backpropagation, the model adjusts its embedding weights to make a better prediction next time.

**The Result:** By repeatedly guessing the next word and correcting its errors, the model naturally updates its embedding weights so that words sharing similar contexts end up with similar mathematical representations.

In [15]:
encode_sample = encode_dataset[100:]

One of the easiest and most intuitive ways to create the input–target pairs for the nextword prediction task is to create two variables, x and y, where x contains the input
tokens and y contains the targets, which are the inputs shifted by 1

In [16]:
context_size = 4
x = encode_sample[:context_size]
y = encode_sample[1:context_size + 1]
print("x:", x)
print("y:      ", y)

x: [198, 198, 41631, 11]
y:       [198, 41631, 11, 484]


In [17]:
for i in range(1,context_size+1):
    context = encode_sample[:i]
    desired = encode_sample[i]
    print(context, "->", desired)

[198] -> 198
[198, 198] -> 41631
[198, 198, 41631] -> 11
[198, 198, 41631, 11] -> 484


Everything left of the arrow (---->) refers to the input an LLM would receive, and
the token ID on the right side of the arrow represents the target token ID that the
LLM is supposed to predict. Let’s repeat the previous code but convert the token IDs
into tex

In [18]:
for i in range(0,context_size+1):
    context = encode_sample[i:i+context_size]
    desired = encode_sample[i+context_size]
    print(tokenizer.decode(context), "----->", tokenizer.decode([desired]))



Together, ----->  they

Together, they ----->  shared
Together, they shared ----->  the
, they shared the ----->  needle
 they shared the needle ----->  and


### Creating Input & Target Sequences

This code converts tokenized text into **training samples for a language model**.

* `input_ids` → tokens given to the model.
* `target_ids` → the same sequence shifted **one token to the right**, representing the **next token to predict**.
* `max_length` → number of tokens in each sequence.
* `stride` → how many tokens to move the sliding window each time.

**Example:**

```text
Tokens:  [1, 2, 3, 4, 5, 6]

Input:   [1, 2, 3, 4]
Target:  [2, 3, 4, 5]
```

The model learns:

```text
1 → 2
2 → 3
3 → 4
4 → 5
```

`stride` controls the overlap between consecutive sequences.

**In short:**

> Tokenized text → sliding windows → input/target pairs → PyTorch tensors → ready for LLM training.


In [19]:
import torch 
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, data, tokenizer, max_length,stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(data)
        for i in range(0,len(token_ids) - max_length,stride):
            input_seq = token_ids[i:i+max_length]
            target_seq = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_seq))
            self.target_ids.append(torch.tensor(target_seq))
            
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

## A data loader to generate batches with input-with pairs

In [20]:
from torch.utils.data import DataLoader

def create_dataloader_v1(data, batch_size = 32,
                         max_length =256,
                         stride = 128,
                         drop_last = True,
                         shuffle = True,
                         num_workers = 0):
    tokenizer = tiktoken.get_encoding("gpt2")  # Initializes the  tokenizer
    dataset = GPTDataset(data, tokenizer, max_length, stride)    # Creates dataset
    dataloader = DataLoader(dataset, 
                            batch_size=batch_size, 
                            shuffle=shuffle,
                            drop_last=drop_last,   # Drops the last batch if it is smaller than the specified batch size to prevent loss spike
                            num_workers=num_workers)
    return dataloader

lets test how our dataloader works 

In [21]:
dataloaders = create_dataloader_v1(dataset_text, batch_size=1, max_length=5, stride=1,shuffle=False)
data_iter = iter(dataloaders)
first_batch = next(data_iter)
print("First batch input_ids:", first_batch)

First batch input_ids: [tensor([[3198, 1110,   11,  257, 1310]]), tensor([[1110,   11,  257, 1310, 2576]])]


In [22]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[1110,   11,  257, 1310, 2576]]), tensor([[  11,  257, 1310, 2576, 3706]])]


If we compare the first and second batches, we can see that the second batch’s token
IDs are shifted by one position . The stride setting dictates the
number of positions the inputs shift across batches, emulating a sliding window
approach

# Creating token embeddings

In [23]:
vocab_size = tokenizer.n_vocab # 50257
output_dimension = 512
token_embedding_layer = torch.nn.Embedding(vocab_size,output_dimension)

In [24]:
max_length = 4
dataloader = create_dataloader_v1(
 dataset_text, batch_size=8, max_length=max_length,
 stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[ 3198,  1110,    11,   257],
        [ 1310,  2576,  3706, 20037],
        [ 1043,   257, 17598,   287],
        [  607,  2119,    13,  1375],
        [ 2993,   340,   373,  2408],
        [  284,   711,   351,   340],
        [  780,   340,   373,  7786],
        [   13, 20037,  2227,   284]])

Inputs shape:
 torch.Size([8, 4])


As we can see, the token ID tensor is 8 × 4 dimensional, meaning that the data batch
consists of eight text samples with four tokens each

In [25]:
token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape)

Token embeddings shape: torch.Size([8, 4, 512])


The 8 × 4 × 512–dimensional tensor output shows that each token ID is now embedded as a 512-dimensional vector.

In [26]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dimension)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape) 

torch.Size([4, 512])


In [35]:
# print single vector
print(f"single vector length : {len(pos_embeddings[0])}")
print("=" * 50)
print(f"first five vectors :{pos_embeddings[:5]}")
print("="*50)

single vector length : 512
first five vectors :tensor([[-0.8344, -1.2643,  0.6492,  ...,  0.4690, -0.0442, -1.3476],
        [ 0.4927, -0.8876, -0.4384,  ...,  0.4662,  0.5883, -1.5186],
        [ 0.7239,  0.1231, -1.0185,  ...,  0.9822, -0.3593, -0.0911],
        [ 0.7627,  0.9242,  0.5301,  ...,  0.3949,  0.4942,  0.1669]],
       grad_fn=<SliceBackward0>)
